In [ ]:
# -*- coding: utf-8 -*-
"""
Detecção Automatizada de Embarcações de Garimpo Ilegal usando Imagens SAR e Redes Neurais Profundas
Autor: Fellipe Machado Castro (Universidade Federal do Pará - UFPA)
"""

print("Instalando bibliotecas necessárias...")
!pip install -q rasterio geopandas segmentation-models-pytorch albumentations torchmetrics

import os, glob, random, shutil
import numpy as np
import rasterio
import geopandas as gpd
from rasterio.features import rasterize
from rasterio.windows import Window
from shapely.geometry import box
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.ndimage import label, find_objects, binary_opening
from scipy.signal.windows import gaussian as gaussian_window
import warnings
warnings.filterwarnings('ignore')

# --- SEMENTE GLOBAL ---
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- CONFIGURAÇÕES DO TCC ---
BASE_DIR = "/content/drive/MyDrive/Dados_TCC_Garimpo"
OUT_CHIPS_DIR = "/content/chips_dataset"
ENCODER_NAME = "resnet34"  # Arquitetura definitiva do TCC
MODEL_SAVE_PATH = os.path.join(BASE_DIR, f"unet_{ENCODER_NAME}.pth")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def normalize_sar(img):
    img = img.astype(np.float32)
    img = np.where(img < -999, np.nan, img)
    img = np.nan_to_num(img, nan=np.nan)
    finite = img[np.isfinite(img)]
    if finite.size == 0: return np.zeros_like(img, dtype=np.float32)
    lo, hi = np.percentile(finite, 2), np.percentile(finite, 99)
    img = np.nan_to_num(img, nan=lo)
    img = np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)
    return img.astype(np.float32)

print(f"Ambiente pronto! Executando com Encoder: {ENCODER_NAME.upper()} no dispositivo: {DEVICE}")

Mounted at /content/drive


In [ ]:
TIF_FILES = glob.glob(os.path.join(BASE_DIR, "imagens_sar_originais", "S1_*.tif"))
CHIP_SIZE = 256
STRIDE = 128
BUFFER_M = 30
SAVE_NEGATIVE_FRACTION = 0.35

if os.path.exists(OUT_CHIPS_DIR): shutil.rmtree(OUT_CHIPS_DIR)
os.makedirs(OUT_CHIPS_DIR, exist_ok=True)

def chip_valido(chip_img, limite_vazio=0.05):
    pixels_pretos = (np.sum(chip_img == 0) + np.sum(np.isnan(chip_img)) + np.sum(chip_img < -999))
    return (pixels_pretos / chip_img.size) <= limite_vazio

def build_mask(tif_path, geojson_path, buffer_m):
    try:
        with rasterio.open(tif_path) as src:
            h, w = src.height, src.width
            bounds = src.bounds
            gdf = gpd.read_file(geojson_path)
            if src.crs: gdf = gdf.to_crs(src.crs)
            bbox = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
            gdf = gdf[gdf.geometry.intersects(bbox)]
            if gdf.empty: return np.zeros((h, w), dtype=np.uint8), 0
            gdf['geometry'] = gdf.geometry.buffer(buffer_m / 111320) if src.crs.is_geographic else gdf.geometry.buffer(buffer_m)
            shapes = ((geom, 1) for geom in gdf.geometry)
            mask = rasterize(shapes, out_shape=(h, w), transform=src.transform, fill=0, dtype=np.uint8)
            return mask, len(gdf)
    except Exception: return np.zeros((100, 100), dtype=np.uint8), 0

print(f"Fatiando {len(TIF_FILES)} cenas orbitais...")
total_chips, chips_descartados = 0, 0

for tif_path in TIF_FILES:
    fname = os.path.basename(tif_path)
    try:
        parts = fname.replace(".tif", "").split("_")
        city, year, month = parts[1], parts[2], parts[3]
    except: continue

    # Busca exclusivamente pelo padrão gabarito_ANO_MES.geojson
    geojson_path = os.path.join(BASE_DIR, f"gabarito_{year}_{month.zfill(2)}.geojson")
    if not os.path.exists(geojson_path):
        geojson_path = os.path.join(BASE_DIR, f"gabarito_{year}_{month}.geojson")
        if not os.path.exists(geojson_path): continue

    mask, n_balsas = build_mask(tif_path, geojson_path, BUFFER_M)
    save_folder = os.path.join(OUT_CHIPS_DIR, f"{city}_{month}")
    os.makedirs(save_folder, exist_ok=True)

    with rasterio.open(tif_path) as src:
        img_data = src.read(1)
        h, w = img_data.shape
        for y in range(0, h - CHIP_SIZE, STRIDE):
            for x in range(0, w - CHIP_SIZE, STRIDE):
                img_chip = img_data[y:y+CHIP_SIZE, x:x+CHIP_SIZE]
                mask_chip = mask[y:y+CHIP_SIZE, x:x+CHIP_SIZE]
                if not chip_valido(img_chip):
                    chips_descartados += 1; continue
                is_pos = (np.max(mask_chip) > 0)
                if not is_pos and random.random() > SAVE_NEGATIVE_FRACTION: continue

                img_norm = normalize_sar(img_chip)
                base_name = f"{city}_{year}_{month}_{y}_{x}"
                np.save(os.path.join(save_folder, base_name + "_img.npy"), img_norm)
                np.save(os.path.join(save_folder, base_name + "_mask.npy"), mask_chip)
                total_chips += 1

print(f"Geração Concluída! Chips úteis: {total_chips}")

Versão do PyTorch: 2.11.0+cu128
Dispositivo de Processamento: CUDA


In [ ]:
BATCH_SIZE = 16
EPOCHS = 80
LR = 3e-4
EARLY_STOP_PATIENCE = 20

def seed_worker(worker_id):
    ws = torch.initial_seed() % 2**32
    np.random.seed(ws); random.seed(ws)

class UnifiedDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.img_files = file_list; self.transform = transform
    def __len__(self): return len(self.img_files)
    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        mask_path = img_path.replace("_img.npy", "_mask.npy")
        image = np.load(img_path).astype(np.float32)[..., None]
        mask  = np.load(mask_path).astype(np.float32)[..., None]
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug['image'], aug['mask']
        return np.transpose(image, (2,0,1)), np.transpose(mask, (2,0,1))

train_tfms = A.Compose([A.RandomRotate90(), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5)])

all_files = [os.path.join(root, f) for root, _, files in os.walk(OUT_CHIPS_DIR) if f.endswith("_img.npy")]
train_files = [f for f in all_files if "_2025_8_" not in f]
val_files   = [f for f in all_files if "_2025_8_" in f]

g = torch.Generator(); g.manual_seed(SEED)
train_dl = DataLoader(UnifiedDataset(train_files, transform=train_tfms), batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g)
val_dl   = DataLoader(UnifiedDataset(val_files, transform=None), batch_size=BATCH_SIZE, shuffle=False)

model = smp.Unet(encoder_name=ENCODER_NAME, encoder_weights="imagenet", in_channels=1, classes=1).to(DEVICE)

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0): super().__init__(); self.s=smooth
    def forward(self, logits, targets):
        p = torch.sigmoid(logits).view(-1); t = targets.view(-1)
        inter = (p*t).sum()
        return 1 - (2*inter + self.s)/(p.sum() + t.sum() + self.s)

bce  = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([10.0]).to(DEVICE))
dice = DiceLoss()
def criterion(yp, yt): return 0.5*bce(yp, yt) + 0.5*dice(yp, yt)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_iou = 0.0; sem_melhora = 0
train_loss_history, val_iou_history = [], []

print("Iniciando Treinamento da U-Net...")
for epoch in range(1, EPOCHS+1):
    model.train(); tl = 0.0; ti = tu = 0
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x); loss = criterion(logits, y)
        loss.backward(); optimizer.step()
        tl += loss.item()
        pr = (torch.sigmoid(logits) > 0.5).int(); yi = y.int()
        ti += (pr & yi).sum().item(); tu += (pr | yi).sum().item()

    model.eval(); vi = vu = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            pr = (torch.sigmoid(logits) > 0.5).int(); yi = y.int()
            vi += (pr & yi).sum().item(); vu += (pr | yi).sum().item()

    scheduler.step()
    avg_tl = tl/len(train_dl); tiou = ti/(tu+1e-6); viou = vi/(vu+1e-6)
    train_loss_history.append(avg_tl); val_iou_history.append(viou)

    if viou > best_iou:
        best_iou = viou; sem_melhora = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
    else:
        sem_melhora += 1
        if sem_melhora >= EARLY_STOP_PATIENCE:
            print(f"Parada antecipada (Early Stopping) na época {epoch}."); break

Estrutura de diretórios configurada.


In [ ]:
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
model.eval()

def predict_tta(model, x):
    outs = []
    for k in range(4):
        xx = torch.rot90(x, k, dims=[2,3])
        with torch.no_grad(): p = torch.sigmoid(model(xx))
        outs.append(torch.rot90(p, -k, dims=[2,3]))
    with torch.no_grad(): p = torch.sigmoid(model(torch.flip(x, dims=[3])))
    outs.append(torch.flip(p, dims=[3]))
    return torch.stack(outs).mean(0)

def _components(mask):
    lbl, n = label(mask); objs=[]
    for i in range(1, n+1):
        ys, xs = np.where(lbl == i)
        if len(ys) >= 8: objs.append(set(zip(ys.tolist(), xs.tolist())))
    return objs

def get_confusion_metrics(probs, tgts, thr):
    TP=FP=FN=0
    for prob, tgt in zip(probs, tgts):
        pred = binary_opening(prob > thr, iterations=1)
        gt = _components(tgt); pr = _components(pred)
        pred_px = set().union(*pr) if pr else set()
        gt_px   = set().union(*gt) if gt else set()
        for gset in gt:
            if gset & pred_px: TP+=1
            else: FN+=1
        for pset in pr:
            if not (pset & gt_px): FP+=1
    P = TP/(TP+FP+1e-6); R = TP/(TP+FN+1e-6); F = 2*P*R/(P+R+1e-6)
    return P, R, F, TP, FP, FN

all_probs, all_tgts = [], []
with torch.no_grad():
    for x, y in val_dl:
        all_probs.append(predict_tta(model, x.to(DEVICE)).cpu().numpy())
        all_tgts.append(y.numpy())
all_probs = np.concatenate(all_probs)[:,0]
all_tgts  = np.concatenate(all_tgts)[:,0].astype(np.uint8)

best_thr, best_f1 = 0.5, -1.0
for thr in np.arange(0.10, 0.91, 0.05):
    P, R, F, _, _, _ = get_confusion_metrics(all_probs, all_tgts, float(thr))
    if F > best_f1: best_f1, best_thr = F, float(thr)

print(f"Limiar de Confiança Otimizado: {best_thr:.2f}")

In [ ]:
RESULT_DIR = os.path.join(BASE_DIR, f"resultados_cena_inteira")
os.makedirs(RESULT_DIR, exist_ok=True)

_g1d = gaussian_window(CHIP_SIZE, std=CHIP_SIZE/4)
GAUSS_KERNEL = np.outer(_g1d, _g1d).astype(np.float32)
GAUSS_KERNEL /= GAUSS_KERNEL.max()

def contar_gabarito(tif_path, base_dir):
    fname = os.path.basename(tif_path)
    try:
        parts = fname.replace(".tif","").split("_"); city, year, month = parts[1], parts[2], parts[3]
    except: return "?"

    # Busca exclusivamente pelo padrão gabarito_ANO_MES.geojson
    gj = os.path.join(base_dir, f"gabarito_{year}_{month.zfill(2)}.geojson")
    if not os.path.exists(gj):
        gj = os.path.join(base_dir, f"gabarito_{year}_{month}.geojson")
        if not os.path.exists(gj): return "?"

    try:
        with rasterio.open(tif_path) as src:
            gdf = gpd.read_file(gj)
            if src.crs: gdf = gdf.to_crs(src.crs)
            return len(gdf[gdf.geometry.intersects(box(*src.bounds))])
    except Exception: return "?"

print("Iniciando reconstrução das cenas orbitais...")

for tif_file in TIF_FILES:
    fname = os.path.basename(tif_file)
    qtd_gab = contar_gabarito(tif_file, BASE_DIR)

    with rasterio.open(tif_file) as src:
        H, W = src.height, src.width
        full_img = src.read(1)
        prob_map   = np.zeros((H,W), dtype=np.float32)
        weight_map = np.zeros((H,W), dtype=np.float32)

        for y in range(0, H, STRIDE):
            for x in range(0, W, STRIDE):
                window = Window(x, y, CHIP_SIZE, CHIP_SIZE).intersection(Window(0,0,W,H))
                chip = src.read(1, window=window)
                if chip.size == 0: continue
                hh, ww = chip.shape
                pad = np.zeros((CHIP_SIZE, CHIP_SIZE), dtype=np.float32)
                pad[:hh,:ww] = chip
                norm = normalize_sar(pad)
                pred = predict_tta(model, torch.from_numpy(norm).unsqueeze(0).unsqueeze(0).to(DEVICE)).cpu().numpy()[0,0]
                ker  = GAUSS_KERNEL[:hh,:ww]
                prob_map[y:y+hh, x:x+ww]   += pred[:hh,:ww]*ker
                weight_map[y:y+hh, x:x+ww] += ker

        prob_map /= (weight_map + 1e-6)
        binary_mask = binary_opening(prob_map > best_thr, iterations=1)
        labeled, n = label(binary_mask)
        objs = find_objects(labeled)
        valid = [o for o in objs if (o[0].stop-o[0].start)*(o[1].stop-o[1].start) >= 12]

        print(f"🛰️ {fname} | Gabarito: {qtd_gab} | Rede Detectou: {len(valid)}")

        plt.figure(figsize=(15,15))
        plt.imshow(normalize_sar(full_img), cmap='gray'); ax = plt.gca()
        for o in valid:
            y0,y1 = o[0].start, o[0].stop; x0,x1 = o[1].start, o[1].stop
            ax.add_patch(patches.Rectangle((x0,y0), x1-x0, y1-y0, linewidth=2, edgecolor='red', facecolor='none'))
        plt.title(f"Detecções na Cena Inteira | {fname} (Limiar={best_thr:.2f})", fontsize=14)
        plt.axis('off')
        plt.savefig(os.path.join(RESULT_DIR, f"CENA_{fname}.png"), bbox_inches='tight', dpi=150); plt.close()

print("Reconstrução finalizada com sucesso!")